# Run RWE on real MIND data (RQ2 / RQ3)

Downloads **MIND-small** from Microsoft's official source (research use — the data is *not* redistributed), ingests it, learns ideological positions from click behaviour, and runs the baselines + RWE-D/RWE-B to print the paper's **RQ2** (accuracy + long-tail) and **RQ3** (ideological-diversity) tables.

Runtime: a few minutes on a free CPU runtime. Nothing is committed — only the printed metrics / `results.csv`.

> If the GitHub repo is **private**, edit the clone URL in the next cell to include a token: `https://<TOKEN>@github.com/greenwichg/random_walks_with_erasure.git`

In [ ]:
# 1) Get the code (branch with the MIND pipeline) and install it
!git clone --branch claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git
%cd random_walks_with_erasure
!pip install -e . -q
print('installed')

In [ ]:
# 1b) Drive cache — make every expensive artifact (the MIND data, lean.csv, the
#     .npz files) survive Colab runtime resets. Mounts Drive once; later cells
#     call cache_get / cache_put, so after the first successful run you never
#     re-download the data or re-run the GPU classifier again.
import os, shutil

CACHE = "/content/drive/MyDrive/rwe_mind"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    CACHE_OK = True
    print("Drive cache ready ->", CACHE)
except Exception as e:
    CACHE_OK = False
    print("(no Drive cache; artifacts will NOT persist across resets):", e)

def cache_get(name):
    """Copy <name> back from the Drive cache into the working dir if available."""
    src = os.path.join(CACHE, os.path.basename(name))
    if CACHE_OK and os.path.exists(src) and not os.path.exists(name):
        shutil.copy(src, name)
        print("restored from Drive cache:", name)
    return os.path.exists(name)

def cache_put(name):
    """Save <name> to the Drive cache for future runs."""
    if CACHE_OK and os.path.exists(name):
        shutil.copy(name, os.path.join(CACHE, os.path.basename(name)))
        print("cached to Drive:", name)

In [ ]:
# 2) Get MIND-small (train). Microsoft GATED the public blob (HTTP 409: "Public
#    access is not permitted"), so the old direct download no longer works for
#    anyone. This cell reuses the Drive cache -> tries the official URL -> falls
#    back to an inline upload, then caches the zip to Drive.
#
#    If it is not already cached, get MINDsmall_train.zip ONCE from a mirror:
#      * Kaggle  - search "MIND microsoft news" (free login), download the train zip
#      * Hugging Face - huggingface.co/datasets, search "MIND"
#    then pick it in the upload dialog this cell pops up.
import os, glob, urllib.request
ZIP = "MINDsmall_train.zip"

def have_news():
    return [h for h in glob.glob("**/news.tsv", recursive=True) if "fixture" not in h]

if not have_news():
    if not cache_get(ZIP):                       # not in cwd and not in Drive cache
        try:
            url = "https://mind201910small.blob.core.windows.net/release/" + ZIP
            print("trying official source (often gated now) ...")
            urllib.request.urlretrieve(url, ZIP)
        except Exception as e:
            print("official source unavailable:", e)
            print("Pick the MINDsmall_train.zip you downloaded from a mirror:")
            from google.colab import files
            up = files.upload()                  # inline file picker
            zips = [f for f in up if f.lower().endswith(".zip")]
            if zips and zips[0] != ZIP:
                os.replace(zips[0], ZIP)
        cache_put(ZIP)                           # persist for future runs
    os.system("unzip -q -o %s -d MINDsmall_train" % ZIP)

assert have_news(), "still no news.tsv - provide MINDsmall_train.zip per the notes above"
print("MIND ready ->", have_news()[0])

In [ ]:
# 3) Ingest: click graph + political tagging + co-click ideological positions
#    (the ideal-point fit is the slow step). Cached to Drive -> skipped on reruns.
#    NOTE: this co-click axis tends to be TOPICAL; the text-lean path (cells 7-8)
#    is the recommended one for the headline numbers.
import glob, os
if not cache_get("mind.npz"):
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    print('using MIND_DIR =', MIND_DIR)
    get_ipython().system(f"python examples/ingest_mind.py --mind-dir {MIND_DIR} --political-only --ideology --min-user-clicks 10 --min-item-clicks 10 --sample-users 15000 --out mind.npz")
    cache_put("mind.npz")
# Watch the printed lean_corr: closer to 1.0 = the latent axis is left-right.

> 📐 **Look at the `AXIS ALIGNMENT` block** in the output below — a `Pearson r` near **+1** and a high **expected-side %** confirm left-leaning users sit on the left (the axis isn't sign-flipped). Copy that number into `docs/RESULTS.md`.

In [ ]:
# 4) Evaluate: baselines + RWE-D/RWE-B -> RQ2 & RQ3 tables + results.csv
#    (remove --no-bprmf to add the slower BPRMF baseline)
!python examples/eval_mind.py --npz mind.npz --out-csv results.csv --no-bprmf

In [ ]:
# 5) Show the full results table and download the CSV
import pandas as pd
df = pd.read_csv('results.csv', index_col=0)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50)
print(df.round(3).to_string())
try:
    from google.colab import files; files.download('results.csv')
except Exception:
    pass

## Bounded-bridging sweep (the extension's key experiment)

Vary RWE-B's *not too far* bound `d` and watch whether bridging stays strong (`uw_shift` high) while recommendations move back toward the centre (`uw_recs` low) instead of the opposite extreme (the `d=inf` row). This is the real-data test of the bounded-bridging idea in `rwe/opinion_dynamics.py`.

In [ ]:
# 6) RWE-B bounded-bridging sweep (reuses mind.npz; no re-ingest)
!python examples/eval_mind.py --npz mind.npz --out-csv sweep.csv \
  --sweep-max-distance 3,2,1.5,1,0.5 --no-bprmf
import pandas as pd
print(pd.read_csv('sweep.csv', index_col=0).round(3).to_string())

## Option B — text-grounded ideology axis (recommended)

The co-click `--ideology` axis above turns out **topical**, not left-right (check the headline eyeball). This section scores each article's lean from its **text** (title + abstract) with a pretrained classifier, uses that as the ideological axis, and re-runs eval + the sweep — so RQ3 is about real lean.

**Switch to a GPU runtime first**: Runtime -> Change runtime type -> GPU.

In [ ]:
# 7) Score each political article's lean from its TEXT (GPU recommended:
#    Runtime -> Change runtime type -> GPU). Cached to Drive, so after a reset
#    this slow step is skipped automatically.
import glob, os
if not cache_get("lean.csv"):
    get_ipython().system("pip install -q transformers")
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/classify_lean.py --mind-dir {MIND_DIR} --political-only --out lean.csv")
    cache_put("lean.csv")
else:
    print("using cached lean.csv (skipped the GPU classifier)")
# eyeball the 'Most LEFT/RIGHT-scored' headlines it prints -- they should look ideological

In [ ]:
# 7b) (optional, GPU) STRONGER AXIS via ENSEMBLE -- average a 2nd independent
#      bias model with the first to cut single-model noise (the codeable lever for
#      a stronger axis; outlet-lean is blocked on MIND's MSN URLs). CHECK the
#      printed id2label of MODEL2 and set LABELS2 to match its label order. Cached.
import glob, os
MODEL2  = "premsa/political-bias-prediction-allsides-BERT"  # any L/C/R bias model
LABELS2 = "-1,0,1"                                          # MUST match MODEL2's id2label
cache_get("lean_b.csv")            # always restore the 2nd-model CSV if it's cached, so
                                   # it lands in the working dir even when lean_ens.csv is
                                   # cached too (# 7c / # 7e need lean_b.csv, not just the ensemble)
if not cache_get("lean_ens.csv"):
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    if not os.path.exists("lean_b.csv"):
        # NOTE the '=' in --label-positions=... : a leading-dash value like -1,0,1
        # must use '=' or argparse reads it as a flag and errors.
        get_ipython().system(f"python examples/classify_lean.py --mind-dir {MIND_DIR} --political-only --model {MODEL2} --label-positions={LABELS2} --out lean_b.csv")
        if os.path.exists("lean_b.csv"):
            cache_put("lean_b.csv")
    if os.path.exists("lean_b.csv"):
        get_ipython().system("python examples/ensemble_lean.py lean.csv lean_b.csv --out lean_ens.csv")
        if os.path.exists("lean_ens.csv"):
            cache_put("lean_ens.csv")
    else:
        print("lean_b.csv was not written -- check the classify_lean.py output above "
              "(is MODEL2 a valid HF id? does --label-positions match its id2label?). "
              "Skipping the ensemble.")
# Read the pairwise Spearman it prints: near 0 = the models disagree, so averaging
# is dubious. To EVALUATE the ensemble axis, ingest with --positions-csv lean_ens.csv
# (in place of lean.csv) and re-run # 8 / # 8b; validate it against your gold set:
#   python examples/validate_lean.py --lean lean_ens.csv --against label_template.tsv


In [ ]:
# 7e) ARTICLE-LEVEL RELIABILITY -- how much do the two independent bias models
#      agree on a SINGLE headline? Spearman + Cohen's kappa on L/C/R buckets +
#      the side-flip rate (one model Left, the other Right). This is the honest
#      "is one article's lean trustworthy, or only the population aggregate?"
#      number a reviewer will ask for. Needs lean.csv (# 7) and lean_b.csv (# 7b).
#      CPU, seconds -- no GPU, no API.
import os
for _f in ("lean.csv", "lean_b.csv"):
    try:
        cache_get(_f)              # pull from the Drive cache if the file is only cached
    except NameError:
        pass                       # cache_get only exists once you've run # 1b
if os.path.exists("lean.csv") and os.path.exists("lean_b.csv"):
    get_ipython().system("python examples/lean_agreement.py lean.csv lean_b.csv --out lean_disagree.csv")
    print("\n>>> Read the kappa: LOW (fair/slight) = the per-article label is noisy, so "
          "the lean axis is only reliable in AGGREGATE (per-user / per-outlet means) -- "
          "state this in the paper, it's the honest counterpart to the RWE numbers "
          "(which ARE aggregate). 'side only' kappa asks the weaker question (Left vs "
          "Right, ignoring the Center boundary). lean_disagree.csv lists the split articles.")
else:
    print("need lean.csv (# 7) and lean_b.csv (# 7b). If they're only in your Drive "
          "cache, run # 1b (mounts Drive) first so this cell can restore them; "
          "otherwise run # 7 and # 7b, then re-run this cell.")

In [ ]:
# 7f) (SPIKE) OUTLET-LEAN via MSN publisher resolution -- tests whether the
#      HIGH-confidence branch of the hybrid axis is even reachable on MIND. MIND hides
#      the publisher behind MSN URLs, but the MIND team published article snapshots at
#      assets.msn.com/labs/mind/<news_id>.html. This fetches a SAMPLE (browser UA +
#      retries -> beats the 409 gating), PARSES the original publisher (JSON-LD /
#      og:site_name / MSN provider JSON / canonical host / byline), and reports how many
#      resolved outlets join examples/data/outlet_lean.csv. Needs OPEN network -- Colab
#      has it; a 403/409 wall means the snapshots are gated and the branch is NOT
#      reachable (which is itself the answer the spike returns). Start SMALL with --limit.
import glob, os
MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
get_ipython().system(f"python examples/resolve_msn_publisher.py --mind-dir {MIND_DIR} --political-only --limit 200 --sleep 0.2 --lean-csv examples/data/outlet_lean.csv --out source_map.tsv")
try:
    if os.path.exists("source_map.tsv") and os.path.getsize("source_map.tsv") > 0:
        cache_put("source_map.tsv")
except NameError:
    pass                                              # cache_put only exists after # 1b
# If snapshots DO resolve, build an OUTLET-lean npz (the high-confidence branch) and
# feed it to the same eval as # 8:
#   !python examples/ingest_mind.py --mind-dir {MIND_DIR} --source-map source_map.tsv \
#       --lean-csv examples/data/outlet_lean.csv --political-only --min-user-clicks 10 \
#       --min-item-clicks 10 --out mind_outlet.npz
print("\n>>> Read 'fetched N/M' + 'lean-join coverage': N=0 => snapshots gated (branch "
      "unreachable -- report that as the finding); high coverage => wire mind_outlet.npz "
      "in as the high-confidence axis. Either way this settles the (b) question empirically.")

In [ ]:
# 7c) Validate the axes against a HUMAN gold set (CPU, seconds) -- the number
#      that decides whether the ENSEMBLE beats the single model. Run once to get a
#      labeling template, fill it, then re-run to get the three Spearman scores.
#      Your labels are cached to Drive so they survive a runtime reset.
import glob, os, csv
GOLD = "label_gold.tsv"          # your news_id<TAB>position<TAB>title file
cache_get(GOLD)                  # restore filled labels from Drive if present

if not os.path.exists(GOLD):
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    # stratify the 40 sample headlines over the ENSEMBLE's range (spans L/C/R)
    get_ipython().system(f"python examples/validate_lean.py --lean lean_ens.csv --news-dir {MIND_DIR} --sample 40 --out {GOLD}")
    print(f"\n>>> STEP 1 done. Open {GOLD} (Colab file browser, left), fill the "
          "'position' column with -1 (left) / 0 (center) / +1 (right) for each "
          "headline -- WITHOUT looking at any model score -- save, then RE-RUN this cell.")
else:
    rows = list(csv.reader(open(GOLD), delimiter="\t"))
    filled = sum(1 for r in rows[1:] if len(r) > 1 and r[1].strip() not in ("", "position"))
    if filled < 5:
        print(f"{GOLD} has {filled} labels -- fill the 'position' column "
              "(-1/0/+1) and re-run this cell.")
    else:
        cache_put(GOLD)          # persist your labels across resets
        print(f"validating all three axes against {filled} human-labeled headlines:\n")
        for name, f in [("single  (politicalBiasBERT)", "lean.csv"),
                        ("2nd     (premsa / AllSides)", "lean_b.csv"),
                        ("ENSEMBLE (the two averaged)", "lean_ens.csv")]:
            if os.path.exists(f):
                print(f"================  {name}  [{f}]  ================")
                get_ipython().system(f"python examples/validate_lean.py --lean {f} --against {GOLD}")
                print()
        print(">>> Compare the Spearman rows: ENSEMBLE > single = the ensemble helped. "
              "Paste all three to fold the result into RESULTS.md / the paper.")


In [ ]:
# 7d) (optional, FREE) AUTOMATED convergent validity -- label the SAME headlines with
#      a SECOND model and correlate. Uses Google's free-tier Gemini: no credit card,
#      get a key at https://aistudio.google.com (Get API key), then add it in Colab
#      Secrets (the key icon, left) named GEMINI_API_KEY with notebook access ON.
#      This is NOT a human gold set: the text axis is itself a language model, so
#      model-vs-model agreement shares lexical bias -- convergent validity, weaker
#      than the human --raters path in # 7c, and kept SEPARATE from it.
#      (To use Claude instead: pip install anthropic; set ANTHROPIC_API_KEY; set
#       PROVIDER="anthropic", MODEL="claude-opus-4-8". Costs ~$0.30. Gemini is free.)
import glob, os
get_ipython().system("pip -q install google-genai")

PROVIDER = "gemini"
MODEL    = "gemini-2.5-flash"   # hit a 503 'high demand'? switch to "gemini-2.0-flash"
                                # or "gemini-2.5-flash-lite" (different, also-free pools)

if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

if not cache_get("llm_labels.csv"):                # cache the labels across resets
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    # blind, stratified 120-headline template (position left blank; the labeler reads
    # ONLY the titles, never the classifier's score -> a true independent second opinion).
    # 120 headlines in batches of 20 = 6 requests, far under the free tier's limits.
    get_ipython().system(f"python examples/validate_lean.py --lean lean.csv --news-dir {MIND_DIR} --sample 120 --out llm_template.tsv")
    get_ipython().system(f"python examples/llm_label.py --provider {PROVIDER} --model {MODEL} --template llm_template.tsv --out llm_labels.csv")
    if os.path.exists("llm_labels.csv"):
        cache_put("llm_labels.csv")

# only validate if labeling actually wrote the file (a 503 spike leaves it absent)
if os.path.exists("llm_labels.csv"):
    get_ipython().system("python examples/validate_lean.py --lean lean.csv --against llm_labels.csv")
    print("\n>>> CONVERGENT VALIDITY (model-vs-model), not human ground truth. A "
          "trustworthy axis number still needs the human --raters path in # 7c. "
          "llm_labels.csv carries a provenance stamp naming the model that labeled it.")
else:
    print("llm_labels.csv was not written -- see the error above. If it was a 503 "
          "'high demand', just re-run this cell (it resumes), or set MODEL to "
          "'gemini-2.0-flash' / 'gemini-2.5-flash-lite' above and re-run.")

In [ ]:
# 8) Re-position from the text lean (no --ideology), then eval + sweep. The .npz
#    is cached to Drive (reset-safe); the fast eval/sweep always run.
import glob, os
if not cache_get("mind_text.npz"):
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/ingest_mind.py --mind-dir {MIND_DIR} --political-only --positions-csv lean.csv --min-user-clicks 10 --min-item-clicks 10 --sample-users 15000 --out mind_text.npz")
    cache_put("mind_text.npz")
get_ipython().system("python examples/eval_mind.py --npz mind_text.npz --out-csv results_text.csv --no-bprmf")
get_ipython().system("python examples/eval_mind.py --npz mind_text.npz --out-csv sweep_text.csv --sweep-max-distance 3,2,1.5,1,0.5 --no-bprmf")
import pandas as pd
print('RESULTS (text-lean axis):')
print(pd.read_csv('results_text.csv', index_col=0).round(3).to_string())
print()
print('SWEEP (text-lean axis):')
print(pd.read_csv('sweep_text.csv', index_col=0).round(3).to_string())

In [ ]:
# 8b) Robustness: average over 7 seeds + Wilcoxon significance vs P3
#     (prints mean +/- std tables and p-values; writes *_std / *_pvalues)
!python examples/eval_mind.py --npz mind_text.npz --seeds 7 --no-bprmf \
  --out-csv results_text_ms.csv

In [ ]:
# 8b2) Per-user PAIRED significance (Wilcoxon across users, seed 0). Complements
#      the across-seed test in 8b: for each eligible user it pairs RWE-D vs the
#      reference recommender on AUC / HR / NDCG, then signed-rank tests the
#      per-user differences -- a much larger n than 7 seeds. CPU, ~1-2 min.
get_ipython().system("python examples/eval_mind.py --npz mind_text.npz --per-user-sig --no-bprmf")


In [ ]:
# 8c) Plot where users and items actually sit on the left<->right scale
#     (left-leaning on the left, right-leaning on the right). Uses the
#     text-lean mind_text.npz; saves + shows axis.png.
get_ipython().system("python examples/plot_axis.py --npz mind_text.npz --out axis.png")
from IPython.display import Image, display
display(Image("axis.png"))
try:
    from google.colab import files; files.download("axis.png")
except Exception:
    pass

In [ ]:
# 8e) (optional, GPU) Reporting-vs-opinion register -> register.csv, for the
#     report's Reporting Ratio. Zero-shot (bart-large-mnli); cached to Drive.
import glob, os
if not cache_get("register.csv"):
    get_ipython().system("pip install -q transformers")
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/classify_register.py --mind-dir {MIND_DIR} --political-only --out register.csv")
    cache_put("register.csv")

In [ ]:
# 8f) (optional, GPU, EXPERIMENTAL) Emotional tone -> emotion.csv, for the
#     report's Attention profile / Emotional Balance. Emotion-from-headline is
#     NOISY -- treat as low-confidence (see docs/HEALTH_REPORT_PLAN.md). Cached.
import glob, os
if not cache_get("emotion.csv"):
    get_ipython().system("pip install -q transformers")
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/classify_emotion.py --mind-dir {MIND_DIR} --political-only --out emotion.csv")
    cache_put("emotion.csv")

In [ ]:
# 8d) Information Health Report -- full reading-diet profile.
#     Built on a FULL-catalog ingest (mind_full.npz: ALL topics, not just the
#     political slice), so Topic Diversity and the political-share context are
#     meaningful. The political-only mind_text.npz collapses every item to one
#     category ("news"), which blanks the whole Variety section.
#     --require-political samples readers who actually read >=min-political
#     political articles, so the Viewpoint / Echo / Open-Mindedness scores
#     populate too (most random MIND users read little political news, so those
#     scores are *legitimately* n/a for them). register/emotion (8e/8f) add
#     Reporting / Emotional / Attention; --behaviors adds Open-Mindedness;
#     --confidence-csv lean.csv confidence-WEIGHTS the viewpoint so ambiguous
#     (low-margin) headlines count less + prints a per-reader axis confidence
#     (re-run # 7 so lean.csv carries the new confidence column; an old 2-column
#     lean.csv is ignored gracefully). NOTE: Source Diversity stays n/a on MIND --
#     the URLs are MSN URLs, so the publisher is not in the data. PoC; see
#     docs/HEALTH_REPORT.md.
import os, glob
MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
if not cache_get("mind_full.npz"):
    get_ipython().system(f"python examples/ingest_mind.py --mind-dir {MIND_DIR} --positions-csv lean.csv --min-user-clicks 10 --min-item-clicks 10 --sample-users 15000 --out mind_full.npz")
    cache_put("mind_full.npz")
extra = ""
if os.path.exists("register.csv"): extra += " --register-csv register.csv"
if os.path.exists("emotion.csv"):  extra += " --emotion-csv emotion.csv"
if os.path.exists("lean.csv"):     extra += " --confidence-csv lean.csv"   # confidence-weight the viewpoint
beh = [h for h in glob.glob("**/behaviors.tsv", recursive=True) if "fixture" not in h]
if beh: extra += f" --behaviors {beh[0]}"
get_ipython().system(f"python examples/health_report.py --npz mind_full.npz --sample 3 --require-political --html health_report.html{extra}")
from IPython.display import HTML, display
display(HTML(open("health_report.html").read()))
try:
    from google.colab import files; files.download("health_report.html")
except Exception:
    pass

In [ ]:
# 8g) GEN-AI NARRATIVE HEALTH REPORT -- the demo-facing layer. An LLM turns the
#      engine-computed metrics from # 8d into a warm, plain-language report + a
#      good-faith STEELMAN of the viewpoint the reader under-consumes, and may
#      recommend real opposite-side headlines from the catalog. The LLM narrates
#      ONLY numbers the engine computed (forbidden to invent stats; a grounding
#      check flags any number it makes up) -- generative where it helps, with a
#      validated engine underneath. Free Gemini; reuses GEMINI_API_KEY from # 7d.
#      Needs mind_full.npz from # 8d above.
import os
get_ipython().system("pip -q install google-genai")
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
# auto-picks the reader with the most filled metrics; add --user N to choose one,
# or --provider anthropic to use Claude. Pass real RWE-B recs via --recs "t1|t2".
get_ipython().system("python examples/narrate_report.py --npz mind_full.npz")

In [ ]:
# 8h) (optional) WEB APP -- the clickable demo artifact. Serves the Information Health
#      Report card + AI narrative + steelman + RWE-B-recommended bridging items + grounding
#      checks at a URL, so you demo a real app, not notebook output. Dependency-free (Python
#      stdlib http.server). Reuses GEMINI_API_KEY from # 7d for the narrative (works without
#      it too). Run # 8f first if you want the Attention profile populated. SAFE TO RE-RUN:
#      it shuts the previous server down and reloads emotion.csv/register.csv -- no port
#      clash. For the VALIDATED-axis viewpoint demo, set NPZ to the Politosphere npz and
#      DOMAIN="reddit".
import sys, time, os, importlib
sys.path.insert(0, "examples")
import app as ihr_app
importlib.reload(ihr_app)            # pick up any git-pulled app.py changes (no kernel restart)

NPZ, DOMAIN, PORT = "mind_full.npz", "news", 8000     # or "politosphere_mi200.npz", "reddit"
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass                                          # runs without a key (no narrative)

# Hold the handle in the NOTEBOOK namespace: a notebook variable survives importlib.reload
# (a module global does not), so a re-run hands the previous server back via prior= and we
# shut it down cleanly instead of orphaning it on the port (the old "Address already in use").
_ihr_server = ihr_app.start_server(NPZ, DOMAIN, "gemini", None, "0.0.0.0", PORT,
                                   prior=globals().get("_ihr_server"))
time.sleep(1.0)
try:
    # iframe embeds the app inline (more reliable than a new window under browser security)
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, height=820)     # the app renders below; click a reader
    print(f"App embedded above on port {PORT}. Re-run this cell anytime to reload data.")
except Exception:
    print(f"App serving at http://127.0.0.1:{PORT} -- open it in your browser.")

## Option C — how ideological is the axis? (validate it)

The text-lean axis is a noisy proxy. Quantify it: sample a few articles, label them yourself (without peeking at the model), and correlate. Or score with a second bias model and correlate the two — see `examples/validate_lean.py`.

In [ ]:
# 9) Make a 40-article labeling template, download it, label offline
import glob, os
MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
!python examples/validate_lean.py --lean lean.csv --news-dir {MIND_DIR} --sample 40 --out label_template.tsv
from google.colab import files; files.download('label_template.tsv')
# fill the 'position' column (-1/0/1), re-upload, then run:
#   !python examples/validate_lean.py --lean lean.csv --against label_template.tsv